[AI 이용 프롬프트]
[gemini가 좀 더 프롬프트 토큰 길이가 김. gpt보다.]

너는 러신머닝 전문가야

앙상블기법중에 랜덤포레스트를 공부하려고 하는데, 
앙상블에 대한 개념을 설명하고
랜덤포레스트의 특징과 장단점을 알려주고
실습을 위한 예제코드를 작성해줘
다음 순서에 따라서 작성

1. 랜덤포레스트의 알고리즘을 쉽게 설명하고
2. 설명된 알고리즘을 검증할 수 있는 코드를 파이썬으로 작성
3. 공개 데이터셋을 이용해서 해당 코드를 적용
4. 랜덤포레스트의 장점이 잘 나타나도록 다른 모델과 비교 테스트 진행하는 코드 작성
5. 각 과정을 쉽게 이해하도록 시각화코드를 파이썬으로 작성
6. 이 모든 과정이 에러없이 동작하도록 다시한번 검증을 거쳐서 최적의 코드로 안내
7. 특히 공개 데이터셋은 실제 다운로드 가능한 사이트인지 한번 더 확인하고 직접 다운할 수 있도록 링크도 알려줘

In [4]:
%pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [5]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
wine_quality = fetch_ucirepo(id=186) 
  
# data (as pandas dataframes) 
X = wine_quality.data.features 
y = wine_quality.data.targets 
  
# metadata 
# print(wine_quality.metadata) 
  
# variable information 
print(wine_quality.variables) 


                    name     role         type demographic  \
0          fixed_acidity  Feature   Continuous        None   
1       volatile_acidity  Feature   Continuous        None   
2            citric_acid  Feature   Continuous        None   
3         residual_sugar  Feature   Continuous        None   
4              chlorides  Feature   Continuous        None   
5    free_sulfur_dioxide  Feature   Continuous        None   
6   total_sulfur_dioxide  Feature   Continuous        None   
7                density  Feature   Continuous        None   
8                     pH  Feature   Continuous        None   
9              sulphates  Feature   Continuous        None   
10               alcohol  Feature   Continuous        None   
11               quality   Target      Integer        None   
12                 color    Other  Categorical        None   

               description units missing_values  
0                     None  None             no  
1                     None  Non

In [6]:
y.value_counts()

quality
6          2836
5          2138
7          1079
4           216
8           193
3            30
9             5
Name: count, dtype: int64

In [7]:
(y >=6).sum(), (y <6).sum()

(quality    4113
 dtype: int64,
 quality    2384
 dtype: int64)

In [8]:
# 와인 품질 분류 : y값이 6이상이면 1 그렇지 않으면 0 
y2 = y.quality.apply(lambda x: 1 if x>=6 else 0 )

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

#데이터 분할
X_train, X_test, y_train, y_test = train_test_split(X, y2, stratify=y2, test_size=0.2, random_state=42 )

In [10]:
# 랜덤포레스트 모델 학습
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [11]:
# 결정트리 모델 학습
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [12]:
# 예측
rf_model_pred = rf_model.predict(X_test)
print(f'랜덤포레스트 {classification_report(y_test, rf_model_pred)}')

dt_model_pred = dt_model.predict(X_test)
print(f'결정트리 {classification_report(y_test, dt_model_pred )}')

# 하기 출력 결과 해석 : 랜덤포레스트의 값이 더 좋다. (f1-score) but, 과적합/ 클래스 불균형 해소 필요

랜덤포레스트               precision    recall  f1-score   support

           0       0.81      0.73      0.77       477
           1       0.85      0.90      0.88       823

    accuracy                           0.84      1300
   macro avg       0.83      0.82      0.82      1300
weighted avg       0.84      0.84      0.84      1300

결정트리               precision    recall  f1-score   support

           0       0.69      0.70      0.70       477
           1       0.83      0.82      0.82       823

    accuracy                           0.78      1300
   macro avg       0.76      0.76      0.76      1300
weighted avg       0.78      0.78      0.78      1300



In [13]:
%pip install imblearn

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# 클래스 불균형 처리 0그룹:477  |  0그룹 : 477 을 up scaling
from imblearn.over_sampling import SMOTE
smote = SMOTE (random_state=42)
X_train_resampled1, y_train_resampled1 = smote.fit_resample(X_train, y_train)

# from imblearn.combine import SMOTEENN
# smote_enn = SMOTEENN(random_state=42)


# 랜덤포레스트 모델 학습
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_resampled1, y_train_resampled1)
# 결정트리 모델 학습
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_resampled1, y_train_resampled1)


# 예측
rf_model_pred = rf_model.predict(X_test)
print(f'랜덤포레스트 {classification_report(y_test, rf_model_pred)}')

dt_model_pred = dt_model.predict(X_test)
print(f'결정트리 {classification_report(y_test, dt_model_pred )}')

랜덤포레스트               precision    recall  f1-score   support

           0       0.76      0.80      0.78       477
           1       0.88      0.86      0.87       823

    accuracy                           0.84      1300
   macro avg       0.82      0.83      0.82      1300
weighted avg       0.84      0.84      0.84      1300

결정트리               precision    recall  f1-score   support

           0       0.67      0.71      0.69       477
           1       0.83      0.80      0.81       823

    accuracy                           0.77      1300
   macro avg       0.75      0.76      0.75      1300
weighted avg       0.77      0.77      0.77      1300



In [15]:
# ====== SMOTEENN ====== 이용


from imblearn.combine import SMOTEENN
smote_enn = SMOTEENN(random_state=42)
X_train_resampled2, y_train_resampled2 = smote_enn.fit_resample(X_train, y_train)

# 랜덤포레스트 모델 학습
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_resampled2, y_train_resampled2)
# 결정트리 모델 학습
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train_resampled2, y_train_resampled2)


# 예측
rf_model_pred = rf_model.predict(X_test)
print(f'랜덤포레스트 {classification_report(y_test, rf_model_pred)}')

dt_model_pred = dt_model.predict(X_test)
print(f'결정트리 {classification_report(y_test, dt_model_pred )}')

랜덤포레스트               precision    recall  f1-score   support

           0       0.62      0.86      0.72       477
           1       0.89      0.70      0.79       823

    accuracy                           0.76      1300
   macro avg       0.76      0.78      0.75      1300
weighted avg       0.79      0.76      0.76      1300

결정트리               precision    recall  f1-score   support

           0       0.58      0.76      0.66       477
           1       0.83      0.68      0.75       823

    accuracy                           0.71      1300
   macro avg       0.71      0.72      0.70      1300
weighted avg       0.74      0.71      0.71      1300

